In [1]:
import os
from huggingface_hub import snapshot_download

model_name = "Qwen/Qwen2.5-3B-Instruct"

save_dir = os.path.join(
    os.getcwd(),
    "models",
    "Qwen2.5-3B-Instruct"
)

os.makedirs(save_dir, exist_ok=True)

print(f"Model will be downloaded: {save_dir}")

if os.path.exists(save_dir) and os.listdir(save_dir):
    print("Model already exists, skipping download.")
else:
    snapshot_download(
        repo_id=model_name,
        local_dir=save_dir,
        local_dir_use_symlinks=False,
        revision="main"
    )
    print("\nDownloaded successfully!")

print(f"Files: {os.listdir(save_dir)}")

Model will be downloaded: /home/yesoytur/APilus/models/Qwen2.5-3B-Instruct


/home/yesoytur/APilus/.venv/lib/python3.11/site-packages/huggingface_hub/utils/_validators.py:206: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `snapshot_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(


Fetching 12 files:   0%|          | 0/12 [00:00<?, ?it/s]


Downloaded successfully!
Files: ['LICENSE', '.cache', 'generation_config.json', 'config.json', '.gitattributes', 'vocab.json', 'tokenizer_config.json', 'model.safetensors.index.json', 'model-00002-of-00002.safetensors', 'model-00001-of-00002.safetensors', 'merges.txt', 'tokenizer.json', 'README.md']


In [2]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch
import os

model_path = os.path.join(
    os.getcwd(),
    "models",
    "Qwen2.5-3B-Instruct"
)

tokenizer = AutoTokenizer.from_pretrained(
    model_path,
    trust_remote_code=True
)

model = AutoModelForCausalLM.from_pretrained(
    model_path,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True
)

print("✅ Model loaded!")

`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

✅ Model loaded!


In [ ]:
from transformers import GenerationConfig
import time
import torch

def ask(
    prompt,
    max_new_tokens=512,
    temperature=0.6,
    top_p=0.95,
    top_k=50,
    repetition_penalty=1.1,
    do_sample=True,
    seed=42,
    verbose=True,
):

    if seed is not None:
        torch.manual_seed(seed)

    messages = [
        {"role": "system", "content": "You are a helpful AI coding assistant."},
        {"role": "user", "content": prompt}
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    model_inputs = tokenizer(
        text,
        return_tensors="pt"
    ).to(model.device)

    input_token_count = model_inputs.input_ids.shape[1]

    gen_config = GenerationConfig(
        max_new_tokens=max_new_tokens,
        do_sample=do_sample,
        temperature=temperature if do_sample else None,
        top_p=top_p if do_sample else None,
        top_k=top_k if do_sample else None,
        repetition_penalty=repetition_penalty,
        pad_token_id=tokenizer.eos_token_id,
    )

    start_time = time.perf_counter()

    generated_ids = model.generate(
        **model_inputs,
        generation_config=gen_config
    )

    elapsed = time.perf_counter() - start_time

    output_ids = generated_ids[0][input_token_count:]

    output_token_count = len(output_ids)

    tokens_per_second = output_token_count / elapsed

    response = tokenizer.decode(
        output_ids,
        skip_special_tokens=True
    ).strip()

    if verbose:

        print(f"\n{'─'*48}")
        print(f"🌡 Temperature        : {temperature}")
        print(f"🎯 Top-p             : {top_p}")
        print(f"🔝 Top-k             : {top_k}")
        print(f"🔁 Repetition        : {repetition_penalty}")
        print(f"🎲 Sampling          : {do_sample} | Seed: {seed}")

        print(f"{'─'*42}")

        print(f"📥 Input tokens      : {input_token_count}")
        print(f"📤 Output tokens     : {output_token_count}")

        print(f"⏱ Generation time   : {elapsed:.2f}s")

        print(f"⚡ Speed             : {tokens_per_second:.1f} tok/s")

        print(f"{'─'*48}\n")

    return response


print("✅ ask() ready!")

✅ ask() ready!


In [6]:
print(ask("Who is Acıbadem Universitys Computer Engineering Department Head?", seed=42))


────────────────────────────────────────────────
🌡 Temperature        : 0.7
🎯 Top-p             : 0.9
🔝 Top-k             : 50
🔁 Repetition        : 1.1
🎲 Sampling          : True | Seed: 42
──────────────────────────────────────────
📥 Input tokens      : 34
📤 Output tokens     : 112
⏱ Generation time   : 3.86s
⚡ Speed             : 29.0 tok/s
────────────────────────────────────────────────

I don't have real-time access to update information about specific individuals or current positions at universities, including the Computer Engineering Department head at Acıbadem University. 

To get this information accurately and up-to-date, you should:

1. Visit the official website of Acıbadem University.
2. Look for the latest announcements or faculty listings on their site.
3. Contact the university directly through their administrative office or email.

If you need more general information about Computer Engineering at Acıbadem University, I'd be happy to help with that!


In [7]:
print(ask("Acıbadem Üniversitesi Bilgisayar Mühendisliği Bölüm başkanı kimdir?", temperature=0.6, seed=42))


────────────────────────────────────────────────
🌡 Temperature        : 0.6
🎯 Top-p             : 0.9
🔝 Top-k             : 50
🔁 Repetition        : 1.1
🎲 Sampling          : True | Seed: 42
──────────────────────────────────────────
📥 Input tokens      : 42
📤 Output tokens     : 85
⏱ Generation time   : 2.24s
⚡ Speed             : 38.0 tok/s
────────────────────────────────────────────────

Üzgünüm, şu anda Acıbadem Üniversitesi'nin Bilgisayar Mühendisliği Bölümünün başkanını gerçek zamanlı bir şekilde bulunamadığımıza karar vermiş oluyorum. Üniversite resmi web sitesi veya iletişim kaynaklarını kontrol etmenizi öneririm veya bilgi almak için üniversiteye doğrudan bir mesaj gönderirsiniz.


In [8]:
print(ask("En iyi Aİ hangisidir?", temperature=0.6, seed=42))


────────────────────────────────────────────────
🌡 Temperature        : 0.6
🎯 Top-p             : 0.9
🔝 Top-k             : 50
🔁 Repetition        : 1.1
🎲 Sampling          : True | Seed: 42
──────────────────────────────────────────
📥 Input tokens      : 31
📤 Output tokens     : 296
⏱ Generation time   : 7.70s
⚡ Speed             : 38.5 tok/s
────────────────────────────────────────────────

Açıklamadan önce bir not: Bilinen tüm sistemlerde en iyi bir AI var olmasa da, bu bir konu sadece kesin bir cevap veremez. 

Ancak genellikle kullanılan bazı popüler ve uzun zamanda performans gösteren AI sistemleri şunlardır:

1. Google's AlphaGo - Oyun yapısı üzerindeki ilerleme

2. IBM Watson - Sağlık bilimine uygulanan AI

3. Tesla Autopilot - Otomobil endüstrisinde AI kullanımı

4. Microsoft's Tay - Genel olarak hatalı sonuçlar verdi ancak ilk deneyimdi

5. Amazon Alexa ve Google Assistant - İos ve Android cihazlarında uygulanan intelligent conversational agents

6. BERT (Bidirectional Encod

In [9]:
print(ask("Sen hangi tarihe kadar eğitildin verilerle", temperature=0.6, seed=42))


────────────────────────────────────────────────
🌡 Temperature        : 0.6
🎯 Top-p             : 0.9
🔝 Top-k             : 50
🔁 Repetition        : 1.1
🎲 Sampling          : True | Seed: 42
──────────────────────────────────────────
📥 Input tokens      : 35
📤 Output tokens     : 72
⏱ Generation time   : 1.89s
⚡ Speed             : 38.0 tok/s
────────────────────────────────────────────────

Üzgünüm, ben bir yapay zeka olmayan bir model olduğum için gerçek zamanlı bilgiye erişimim yok ve eğitim süresi hakkında bilgi sahibi değilim. Tüketici adımlarıyla çalıştığım programlar için geliştiricilere başvurmanız gerekebilir.


In [10]:
print(ask("En iyi Aİ hangisidir?", temperature=0.6, seed=42, top_p=0.95))



────────────────────────────────────────────────
🌡 Temperature        : 0.6
🎯 Top-p             : 0.95
🔝 Top-k             : 50
🔁 Repetition        : 1.1
🎲 Sampling          : True | Seed: 42
──────────────────────────────────────────
📥 Input tokens      : 31
📤 Output tokens     : 281
⏱ Generation time   : 7.29s
⚡ Speed             : 38.6 tok/s
────────────────────────────────────────────────

Açıklamadan önce bir not: Bilinen tüm sistemlerde en iyi bir AI var olmasa da, bu bir tür sorgu için çok karmaşık ve kesin bir cevap veremezsiniz. AI'nın performansı sürekli gelişmekte ve farklı alanda diğerleriyle karşılaştırıldığında yarışmaktadır.

Ancak genel olarak:

1. Google's AlphaFold - Protein modellenme

2. Anthropic's Claude - Sözlü mesajlama

3. DeepMind's AlphGo - Taş oyunları (örneğin; Çatma)

4. Microsoft's Turing Natural Language Generation - Yaratıcı metin üretimi

5. Hugging Face's Transformers - NLP problemleri (örneğin; makine öğrenimi modeli yakalama)

Bu listeye dahil edil